In [3]:
import random
import pandas as pd
from collections import Counter

# Define intents and patient message templates 
intents = {
    "discuss_symptoms": [
        "I've had a persistent headache for three days now with blurred vision.",
        "There's sharp pain in my lower right abdomen that comes and goes.",
        "I'm experiencing shortness of breath even with minimal activity.",
        "I have severe diarrhea and vomiting since last night - can't keep anything down.",
        "My joints are swollen and painful, especially in the mornings.",
        "I noticed blood in my urine this morning with some burning sensation.",
        "I've been having chest palpitations that last several minutes at a time.",
        "There's a strange numbness in my left arm that won't go away."
    ],
    "request_appointment": [
        "I need to schedule a physical exam before my overseas trip next month.",
        "Can I book a follow-up with Dr. Smith for my post-op check?",
        "I'd like to make a same-day urgent appointment if possible.",
        "Are there any weekend slots available for a general consultation?",
        "I need to schedule my annual diabetes screening and checkup.",
        "Could I get an appointment with the cardiology department?",
        "I'd prefer a telehealth visit if available - when's the next opening?",
        "My child needs a sports physical - do you have pediatric appointments tomorrow?"
    ],
    "ask_prescription": [
        "My insulin prescription is running low - can you send a refill to CVS?",
        "I'm traveling abroad and need a 3-month supply of my blood thinners.",
        "The antidepressant dosage doesn't seem effective anymore - can we adjust?",
        "I lost my prescription slip for the antibiotics - can you reissue?",
        "My pharmacy says they need prior authorization for my new cholesterol meds.",
        "I'm having bad side effects from the new medication - can we switch?",
        "Do I need a new prescription for my ongoing thyroid medication?",
        "Can you prescribe something stronger for my chronic back pain?"
    ],
    "follow_up": [
        "It's been 2 weeks since my surgery - when should I come for a wound check?",
        "The antibiotics helped but my sinus infection isn't completely gone.",
        "My blood sugar levels have stabilized - do I still need the same dosage?",
        "The physical therapy exercises caused more pain - is this normal?",
        "I completed the treatment but still feel fatigued - what next?",
        "The biopsy results came back - can we discuss them at my next visit?",
        "I've been tracking my blood pressure as instructed - should I continue the meds?",
        "The rash has spread despite using the prescribed cream - what now?"
    ],
    "complaint": [
        "I've been waiting 45 minutes past my scheduled appointment time.",
        "The receptionist was very rude when I called to reschedule.",
        "The billing department charged me for services I never received.",
        "My test results were supposed to be ready yesterday but still aren't available.",
        "The doctor didn't explain my diagnosis clearly - I have many unanswered questions.",
        "The prescription you sent was out of stock at my pharmacy with no alternatives.",
        "Your online portal isn't working and I can't access my medical records.",
        "The nurse didn't return my urgent call about medication side effects."
    ],
    "general_questions": [
        "What are your clinic hours on holidays?",
        "Do you accept my new insurance plan?",
        "Where can I park when coming for my appointment?",
        "What COVID safety measures are you currently following?",
        "How do I get a copy of my medical records transferred to another provider?",
        "What's your policy on late cancellations?",
        "Which hospital are you affiliated with for emergencies?",
        "Do you provide interpreter services for non-English speakers?"
    ]
}

# Define doctor responses per intent with sentiment tagging
doctor_responses = {
    "discuss_symptoms": {
        "positive": [
            "I understand your concern and I'm glad you came in. Based on what you're describing, we have several good treatment options.",
            "Thank you for providing such clear details about your symptoms. This gives us a promising starting point for treatment."
        ],
        "neutral": [
            "I understand this must be worrying. How long exactly have you been experiencing these symptoms?",
            "That sounds notable. Have you noticed any triggers or patterns to when this occurs?"
        ],
        "negative": [
            "These symptoms are concerning and require immediate attention. I'm not satisfied with what I'm hearing.",
            "I'm quite worried about these symptoms. They indicate a potentially serious condition we need to address."
        ],
        "urgent": [
            "Given what you've described, we need to run tests immediately. This requires urgent attention.",
            "These symptoms need immediate investigation. I'm going to expedite some tests right away."
        ],
        "formal": [
            "Upon initial assessment, your described symptoms suggest several potential diagnoses that warrant further investigation.",
            "According to clinical guidelines, these symptoms necessitate a comprehensive examination and possibly diagnostic imaging."
        ],
        "friendly": [
            "I hear you're really going through a tough time with these symptoms. Let's figure this out together, okay?",
            "You know, I had another patient with similar issues last week - don't worry, we'll get to the bottom of this!"
        ],
        "concerned": [
            "I'm genuinely concerned about what you're experiencing. Let's take these symptoms very seriously.",
            "These symptoms raise some red flags for me. I want to make sure we don't miss anything important."
        ],
        "apologetic": [
            "I'm sorry you've been dealing with this discomfort. We should have caught this during your last visit.",
            "I apologize that you've been suffering. The healthcare system can sometimes miss these early signs."
        ]
    },
    "request_appointment": {
        "positive": [
            "Great news! We have several convenient openings that should work perfectly for your schedule.",
            "I'd be happy to schedule that for you! We have some excellent time slots available."
        ],
        "neutral": [
            "I can help with that. Would you prefer an early morning or late afternoon appointment?",
            "Let me check our availability. Are you looking for something this week or next?"
        ],
        "negative": [
            "Unfortunately, we're extremely booked for the next three weeks due to staff shortages.",
            "I regret to inform you that Dr. Smith's schedule is completely full for the next month."
        ],
        "urgent": [
            "Given your situation, I'll rearrange the schedule to fit you in today. This can't wait.",
            "I'm putting you down for our emergency slot at 3pm today. Please arrive 15 minutes early."
        ],
        "formal": [
            "Upon reviewing the physician's calendar, there is availability on Thursday at 2:15pm or Friday at 10:30am.",
            "The scheduling protocol allows for a consultation on the requested date pending insurance verification."
        ],
        "friendly": [
            "Hey, no problem at all! I can totally squeeze you in this Friday - how does that sound?",
            "You know what, I remember you from your last visit! Let's find a perfect time that works for you."
        ],
        "concerned": [
            "Based on what you're describing, I think we should get you in sooner rather than later. How's tomorrow?",
            "I'm a bit worried about waiting too long for this appointment. Let me see what I can do to expedite it."
        ],
        "apologetic": [
            "I'm so sorry about the wait times lately. Let me do my best to find something that works for you.",
            "I apologize for the scheduling difficulties. We're working to improve our availability."
        ]
    },
    "ask_prescription": {
        "positive": [
            "I'm pleased to renew this prescription for you. The treatment seems to be working very well.",
            "Great news! Your lab results look excellent, so I'm happy to continue your current medication."
        ],
        "neutral": [
            "I can certainly renew that. Has your current dosage been effective for you?",
            "Let me verify - you need a refill for your medication, correct?"
        ],
        "negative": [
            "I'm concerned about renewing this medication without seeing you first. These side effects are troubling.",
            "Unfortunately, I cannot authorize this refill until we address these concerning lab results."
        ],
        "urgent": [
            "This is critical - I'm sending an emergency prescription to your pharmacy to be ready within the hour.",
            "Given your situation, I'm authorizing an immediate refill that you can pick up today."
        ],
        "formal": [
            "Upon review of your medical history, a prescription renewal is authorized pending pharmacy verification.",
            "The clinical protocol indicates that medication adjustment may be warranted based on reported efficacy."
        ],
        "friendly": [
            "No problem at all with that refill! I'll take care of it right away so you don't have to worry.",
            "Hey, thanks for the heads-up about running low! I'll get that sent over to your pharmacy today."
        ],
        "concerned": [
            "I'm a bit worried about these side effects you're mentioning. Let's discuss alternatives to this medication.",
            "These symptoms concern me - before renewing, I'd like to check if this medication is still right for you."
        ],
        "apologetic": [
            "I apologize for the confusion with your prescription. Let me fix this right away.",
            "I'm sorry you've been experiencing these medication issues. That must be frustrating."
        ]
    },
    "follow_up": {
        "positive": [
            "I'm very encouraged by your progress! The treatment is working exactly as we hoped.",
            "These are excellent results. You're responding to treatment better than expected."
        ],
        "neutral": [
            "Glad you're following up. How have your symptoms progressed since we last spoke?",
            "Let's review your case. What improvements have you noticed with the treatment?"
        ],
        "negative": [
            "These test results are not what we hoped for. We need to reconsider our approach immediately.",
            "I'm not satisfied with your progress. These symptoms indicate the treatment isn't working."
        ],
        "urgent": [
            "Given what you're describing, I need you to come in immediately for reassessment.",
            "These follow-up symptoms require urgent attention. Can you come to the office today?"
        ],
        "formal": [
            "Upon review of your follow-up data, clinical parameters suggest modification of the treatment protocol.",
            "The post-procedure assessment indicates satisfactory healing with no contraindications for resuming normal activity."
        ],
        "friendly": [
            "Hey there! So good to hear from you! How are you feeling after our last appointment?",
            "You know, I've been thinking about your case - sounds like you're really making progress!"
        ],
        "concerned": [
            "I'm actually quite worried about these lingering symptoms. They shouldn't still be present.",
            "These test results raise some concerns for me. Let's take a closer look at what's happening."
        ],
        "apologetic": [
            "I'm sorry the treatment hasn't provided the relief we expected. Let's try a different approach.",
            "I apologize that you're still experiencing these symptoms. We should have addressed this differently."
        ]
    },
    "complaint": {
        "positive": [
            "Thank you so much for this feedback. It's patients like you who help us improve our services.",
            "I appreciate you bringing this to our attention. We're always looking for ways to do better."
        ],
        "neutral": [
            "I understand your concern. Let me look into this matter for you.",
            "Thank you for bringing this to our attention. This isn't our usual standard."
        ],
        "negative": [
            "This is a serious failure on our part. I'm deeply troubled by what you've experienced.",
            "This level of service is completely unacceptable. I share your frustration entirely."
        ],
        "urgent": [
            "This requires immediate attention. I'm escalating this to our director right now.",
            "Given the severity of this issue, I'm taking immediate steps to address it today."
        ],
        "formal": [
            "Your grievance has been documented and will be addressed according to our patient resolution protocol.",
            "The administrative team will conduct a thorough investigation into the reported procedural deviation."
        ],
        "friendly": [
            "Oh no! That definitely wasn't supposed to happen! Let me help make this right for you.",
            "I'm totally with you on this one - that experience wasn't okay, and we can do better!"
        ],
        "concerned": [
            "I'm deeply troubled by what you experienced. This isn't the care standard we aim to provide.",
            "Your experience worries me greatly. No patient should face these kinds of issues."
        ],
        "apologetic": [
            "I sincerely apologize for this experience - it's completely our fault and we will make it right.",
            "I am so sorry this happened to you. There's no excuse for this kind of treatment."
        ]
    },
    "general_questions": {
        "positive": [
            "Great question! We're pleased to offer extended holiday hours for your convenience.",
            "I'm happy to confirm that we accept your insurance with excellent coverage options."
        ],
        "neutral": [
            "Our regular hours are 8am-6pm weekdays, with urgent care until 8pm.",
            "We accept most major insurance plans. May I have your member ID to verify?"
        ],
        "negative": [
            "Unfortunately, our holiday hours are significantly reduced due to staffing limitations.",
            "I regret to inform you that we no longer accept that insurance plan as of last month."
        ],
        "urgent": [
            "This information is critical - our emergency services remain open 24/7 including all holidays.",
            "For urgent needs, please note our special holiday triage line is always available."
        ],
        "formal": [
            "The facility operational hours during designated holidays are restricted to 10am-2pm for essential services only.",
            "Insurance verification protocols require submission of credentials 48 hours prior to scheduled appointments."
        ],
        "friendly": [
            "Oh, holidays are actually when we have our most flexible hours! We try to make it super convenient!",
            "You know what, parking is actually free on weekends - a little bonus for our Saturday patients!"
        ],
        "concerned": [
            "I'm worried you might have trouble with parking during construction - let me suggest some alternatives.",
            "I'm concerned about your insurance coverage. Let me check some details to make sure you're fully covered."
        ],
        "apologetic": [
            "I apologize that our holiday schedule is so limited. I know it can make things difficult.",
            "I'm sorry about the confusion regarding our policies. We need to communicate these better."
        ]
    }
}

# Define sentiment associations with intents (probability weights)
# This creates more realistic sentiment distribution while allowing for balance
sentiment_intent_weights = {
    "discuss_symptoms": {
        "positive": 0.05, "neutral": 0.15, "negative": 0.2, 
        "urgent": 0.2, "formal": 0.05, "friendly": 0.05, 
        "concerned": 0.25, "apologetic": 0.05
    },
    "request_appointment": {
        "positive": 0.2, "neutral": 0.3, "negative": 0.1, 
        "urgent": 0.05, "formal": 0.15, "friendly": 0.1, 
        "concerned": 0.05, "apologetic": 0.05
    },
    "ask_prescription": {
        "positive": 0.15, "neutral": 0.25, "negative": 0.1, 
        "urgent": 0.1, "formal": 0.15, "friendly": 0.1, 
        "concerned": 0.1, "apologetic": 0.05
    },
    "follow_up": {
        "positive": 0.2, "neutral": 0.2, "negative": 0.1, 
        "urgent": 0.05, "formal": 0.1, "friendly": 0.15, 
        "concerned": 0.15, "apologetic": 0.05
    },
    "complaint": {
        "positive": 0.05, "neutral": 0.1, "negative": 0.15, 
        "urgent": 0.1, "formal": 0.1, "friendly": 0.05, 
        "concerned": 0.15, "apologetic": 0.3
    },
    "general_questions": {
        "positive": 0.15, "neutral": 0.4, "negative": 0.05, 
        "urgent": 0.05, "formal": 0.15, "friendly": 0.1, 
        "concerned": 0.05, "apologetic": 0.05
    }
}

sentiments = ["positive", "neutral", "negative", "urgent", "formal", "friendly", "concerned", "apologetic"]

# Simple summary generator
def generate_summary(intent, sentiment):
    intent_summaries = {
        "discuss_symptoms": "Patient described current health symptoms.",
        "request_appointment": "Patient requested a future consultation.",
        "ask_prescription": "Patient asked for medication renewal or advice.",
        "follow_up": "Patient followed up regarding previous consultation.",
        "complaint": "Patient reported dissatisfaction or issues.",
        "general_questions": "Patient asked general information about services."
    }
    
    sentiment_modifiers = {
        "positive": "Communication had an optimistic tone.",
        "neutral": "Communication was matter-of-fact.",
        "negative": "Communication expressed concerns or dissatisfaction.",
        "urgent": "Communication emphasized immediate action needed.",
        "formal": "Communication was professional and technical.",
        "friendly": "Communication was warm and personable.",
        "concerned": "Communication expressed worry or caution.",
        "apologetic": "Communication included regret or apologies."
    }
    
    return f"{intent_summaries.get(intent)} {sentiment_modifiers.get(sentiment)}"

# Enhanced key point extractor
def extract_key_points(message, intent, sentiment):
    # Get 3-5 most relevant words based on intent and sentiment
    important_words = [word for word in message.split() if len(word) > 4]
    
    # Add intent and sentiment specific extractions
    if intent == "discuss_symptoms":
        body_parts = ["head", "chest", "arm", "leg", "back", "stomach", "throat", "abdomen", "joint"]
        symptoms = ["pain", "ache", "fever", "cough", "rash", "swelling", "numbness", "dizziness"]
        for word in body_parts + symptoms:
            if word in message.lower() and word not in important_words:
                important_words.append(word)
    
    if sentiment == "urgent":
        urgent_words = ["immediate", "urgent", "emergency", "severe", "critical"]
        for word in urgent_words:
            if word in message.lower() and word not in important_words:
                important_words.append(word)
    
    # Take 3-5 words depending on message length
    max_words = min(5, max(3, len(important_words)))
    return ", ".join(important_words[:max_words])

# Balanced data generation function
def generate_balanced_data(num_samples=500, distribution='balanced'):
    data = []
    conversation_id = 1
    
    # Initialize counters for balanced distribution
    intent_counter = {intent: 0 for intent in intents.keys()}
    sentiment_counter = {sentiment: 0 for sentiment in sentiments}
    
    # Calculate targets for balanced distribution
    intent_target = num_samples // len(intents)
    sentiment_target = num_samples // len(sentiments)
    
    # For each intent
    for intent in intents.keys():
        # For each sentiment
        for sentiment in sentiments:
            # Calculate how many samples of this intent-sentiment pair we should create
            if distribution == 'balanced':
                # Completely balanced across all combinations
                target = num_samples // (len(intents) * len(sentiments))
            elif distribution == 'realistic':
                # Use weighted distribution but ensure minimum representation
                intent_portion = num_samples // len(intents)
                target = max(5, int(intent_portion * sentiment_intent_weights[intent][sentiment]))
            else:
                # Default balanced
                target = num_samples // (len(intents) * len(sentiments))
            
            # Generate samples for this intent-sentiment pair
            for _ in range(target):
                patient_msg = random.choice(intents[intent])
                
                # Ensure we have responses for this sentiment
                if sentiment in doctor_responses[intent]:
                    doctor_reply = random.choice(doctor_responses[intent][sentiment])
                else:
                    # Fallback to neutral if specific sentiment not available
                    doctor_reply = random.choice(doctor_responses[intent]["neutral"])
                
                summary = generate_summary(intent, sentiment)
                key_points = extract_key_points(patient_msg, intent, sentiment)
                
                data.append({
                    "conversation_id": conversation_id,
                    "patient_message": patient_msg,
                    "doctor_response": doctor_reply,
                    "intent": intent,
                    "sentiment": sentiment,
                    "summary": summary,
                    "key_points": key_points
                })
                
                conversation_id += 1
                intent_counter[intent] += 1
                sentiment_counter[sentiment] += 1
    
    # If we need to adjust to exactly hit our target number
    while len(data) < num_samples:
        # Find underrepresented intent and sentiment
        min_intent = min(intent_counter, key=intent_counter.get)
        min_sentiment = min(sentiment_counter, key=sentiment_counter.get)
        
        patient_msg = random.choice(intents[min_intent])
        doctor_reply = random.choice(doctor_responses[min_intent][min_sentiment])
        summary = generate_summary(min_intent, min_sentiment)
        key_points = extract_key_points(patient_msg, min_intent, min_sentiment)
        
        data.append({
            "conversation_id": conversation_id,
            "patient_message": patient_msg,
            "doctor_response": doctor_reply,
            "intent": min_intent,
            "sentiment": min_sentiment,
            "summary": summary,
            "key_points": key_points
        })
        
        conversation_id += 1
        intent_counter[min_intent] += 1
        sentiment_counter[min_sentiment] += 1
    
    # If we have too many, trim evenly
    if len(data) > num_samples:
        data = data[:num_samples]
    
    # Shuffle the data to prevent any sequence bias
    random.shuffle(data)
    
    # Reset conversation IDs after shuffling
    for i, entry in enumerate(data):
        entry["conversation_id"] = i + 1
    
    return pd.DataFrame(data)

# Generate and save the data
def main():
    # Generate balanced dataset
    df_balanced = generate_balanced_data(500, distribution='balanced')
    df_balanced.to_csv("balanced_synthetic_chat_data.csv", index=False)
    print("✅ Balanced synthetic chat data saved to 'balanced_synthetic_chat_data.csv'")
    
    # Generate realistic but still balanced dataset
    df_realistic = generate_balanced_data(500, distribution='realistic')
    df_realistic.to_csv("realistic_balanced_chat_data.csv", index=False)
    print("✅ Realistic balanced synthetic chat data saved to 'realistic_balanced_chat_data.csv'")
    
    # Print distribution statistics
    print("\nIntent Distribution (Balanced):")
    print(df_balanced['intent'].value_counts())
    
    print("\nSentiment Distribution (Balanced):")
    print(df_balanced['sentiment'].value_counts())
    
    print("\nIntent Distribution (Realistic):")
    print(df_realistic['intent'].value_counts())
    
    print("\nSentiment Distribution (Realistic):")
    print(df_realistic['sentiment'].value_counts())

if __name__ == "__main__":
    main()

✅ Balanced synthetic chat data saved to 'balanced_synthetic_chat_data.csv'
✅ Realistic balanced synthetic chat data saved to 'realistic_balanced_chat_data.csv'

Intent Distribution (Balanced):
intent
request_appointment    84
discuss_symptoms       84
ask_prescription       83
general_questions      83
complaint              83
follow_up              83
Name: count, dtype: int64

Sentiment Distribution (Balanced):
sentiment
neutral       63
negative      63
urgent        63
positive      63
formal        62
apologetic    62
concerned     62
friendly      62
Name: count, dtype: int64

Intent Distribution (Realistic):
intent
general_questions      85
discuss_symptoms       84
ask_prescription       83
request_appointment    83
follow_up              83
complaint              82
Name: count, dtype: int64

Sentiment Distribution (Realistic):
sentiment
neutral       113
positive       66
concerned      62
formal         57
negative       57
apologetic     49
friendly       48
urgent        